# 22 · WGFMU wake-up · L0 干跑 (dryrun)

**目的**：不连机，纯 Python 验证 `WakeupStage` × `WakeupReadout` 构造的 segments 正确，pgm/ers/read 三段交替结构成立。

**通过标准**：
1. 100 cycles × 3 segs/cycle = 300 segments
2. 只有 readout 段有 measure_event (100 个)
3. 前 5 cycle 波形肉眼看到 pgm↓ ers↑ read↓ 交替
4. assertion 全过


In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import matplotlib.pyplot as plt
import json

from fefetlab.measurements.wgfmu.wakeup import (
    WakeupStage, WakeupReadout, _build_wakeup_segments, WgfmuWakeupConfig
)
from fefetlab.measurements.wgfmu.pulse_builder import PulseTrainBuilder

print("imports OK")


## 1. 构造 2 个 stage (FeFET wake-up 力度阶梯)


In [ ]:
stages = [
    WakeupStage(n_cycles=50, v_pgm=-3.0, v_ers=+3.0,
                t_pgm_s=30e-6, t_ers_s=30e-6,
                rise_fall_s=1e-6, inter_pulse_s=2e-6,
                label="weak_3V"),
    WakeupStage(n_cycles=50, v_pgm=-5.0, v_ers=+5.0,
                t_pgm_s=30e-6, t_ers_s=30e-6,
                rise_fall_s=1e-6, inter_pulse_s=2e-6,
                label="strong_5V"),
]

readout = WakeupReadout(
    v_read=-0.5,
    t_read_s=5e-6,
    rise_fall_s=1e-6,
    measure_points=10,
    measure_average_s=200e-9,
)

print(f"stages: {len(stages)}")
print(f"total cycles: {sum(s.n_cycles for s in stages)}")


## 2. 构造 segments


In [ ]:
segments, cycle_meta = _build_wakeup_segments(stages, readout)

print(f"len(segments)   : {len(segments)}  (期望 300 = (50+50)*3)")
print(f"len(cycle_meta) : {len(cycle_meta)}  (期望 100)")

# segment 类型分布
labels_pgm  = sum(1 for s in segments if s.label.endswith("_pgm"))
labels_ers  = sum(1 for s in segments if s.label.endswith("_ers"))
labels_read = sum(1 for s in segments if s.label.endswith("_read"))
print(f"pgm / ers / read: {labels_pgm} / {labels_ers} / {labels_read}")

# 只有 read 段有 measure
n_with_measure = sum(1 for s in segments if s.measure_during_high and s.measure_points > 0)
print(f"segments with measure window: {n_with_measure} (应=100, 即所有 read)")


## 3. 编译为 plan


In [ ]:
builder = PulseTrainBuilder(pattern_name="dryrun_wakeup_pattern", v_init=0.0, v_base=0.0)
plan = builder.build(segments)

print(f"plan.n_segments       : {len(plan.segments)}")
print(f"plan.n_vectors        : {len(plan.vectors)}")
print(f"plan.n_measure_events : {len(plan.measure_events)}")
print(f"plan.total_duration_s : {plan.total_duration_s*1e3:.3f} ms")


## 4. 画前 5 个 cycle 的波形 (肉眼看 pgm↓ ers↑ read↓ 交替)


In [ ]:
# 取前 5 cycle = 前 15 个 segments
n_show_cycles = 5
n_show_segs = n_show_cycles * 3
t_end_show = plan.segments[n_show_segs-1]["t_end_s"]

t, v = plan.waveform_samples(dt_s=5e-7)
mask = t <= t_end_show + 1e-9

fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(t[mask]*1e6, v[mask], lw=1.0, color="#1f77b4")
ax.set_xlabel("time (us)")
ax.set_ylabel("voltage (V)")
ax.set_title(f"Dryrun wake-up · first {n_show_cycles} cycles (stage 'weak_3V')")
ax.grid(alpha=0.3)
ax.axhline(0, color="gray", lw=0.5)

# 标 read 窗
for seg in plan.segments[:n_show_segs]:
    if seg["measure_points"] > 0:
        ax.axvspan(seg["t_high_start_s"]*1e6, seg["t_high_end_s"]*1e6,
                   alpha=0.25, color="green")

ax.text(0.02, 0.95, "green = read measure window", transform=ax.transAxes,
        va="top", fontsize=9, bbox=dict(boxstyle="round", facecolor="white", alpha=0.8))
plt.tight_layout()
plt.show()


## 5. assertion


In [ ]:
errors = []

# 5.1 segment 总数
expected = sum(s.n_cycles for s in stages) * 3
if len(segments) != expected:
    errors.append(f"segments: expect {expected}, got {len(segments)}")

# 5.2 cycle_meta
if len(cycle_meta) != sum(s.n_cycles for s in stages):
    errors.append(f"cycle_meta: expect {sum(s.n_cycles for s in stages)}, got {len(cycle_meta)}")

# 5.3 measure event 数 = read 段数
read_segs = [s for s in plan.segments if s["measure_points"] > 0]
if len(plan.measure_events) != len(read_segs):
    errors.append(f"measure_events {len(plan.measure_events)} != read_segs {len(read_segs)}")
if len(read_segs) != sum(s.n_cycles for s in stages):
    errors.append(f"read_segs {len(read_segs)} != total_cycles {sum(s.n_cycles for s in stages)}")

# 5.4 pgm/ers 段不应有 measure
for s in plan.segments:
    if s["label"].endswith(("_pgm", "_ers")) and s["measure_points"] > 0:
        errors.append(f"unexpected measure on {s['label']}")
        break

# 5.5 stage 切换处 v_pgm 应该变化
v_pgm_seq = [c["v_pgm"] for c in cycle_meta]
if v_pgm_seq[49] == v_pgm_seq[50]:
    errors.append(f"stage transition not visible: cycle 49={v_pgm_seq[49]}, cycle 50={v_pgm_seq[50]}")

if errors:
    print("FAIL")
    for e in errors: print(" -", e)
    raise AssertionError(f"{len(errors)} wakeup dryrun failure(s)")
else:
    print(f"PASS · {len(segments)} segments, {len(plan.measure_events)} read windows, stage transition OK")


## 6. dump plan.json


In [ ]:
out_dir = ROOT / "notebooks" / "_dryrun_out"
out_dir.mkdir(parents=True, exist_ok=True)
out_path = out_dir / "wakeup_plan.json"

out_path.write_text(json.dumps({
    "pattern_name": plan.pattern_name,
    "v_init": plan.v_init,
    "v_base": plan.v_base,
    "total_duration_s": plan.total_duration_s,
    "n_vectors": len(plan.vectors),
    "n_measure_events": len(plan.measure_events),
    "n_segments": len(plan.segments),
    "stages": [{"n_cycles": s.n_cycles, "v_pgm": s.v_pgm, "v_ers": s.v_ers,
                "t_pgm_s": s.t_pgm_s, "t_ers_s": s.t_ers_s, "label": s.label} for s in stages],
    "readout": {"v_read": readout.v_read, "t_read_s": readout.t_read_s,
                "measure_points": readout.measure_points},
    "cycles_preview_first5": cycle_meta[:5],
    "cycles_preview_last5": cycle_meta[-5:],
}, ensure_ascii=False, indent=2), encoding="utf-8")

print(f"dumped: {out_path}")
print(f"size: {out_path.stat().st_size} bytes")


---

## 通过标准回顾

- [ ] 第 4 步图看到 5 个完整 cycle，每 cycle 三段 (pgm 向下、ers 向上、read 小幅向下)，green 窗只在 read 上
- [ ] 第 5 步 `PASS`
- [ ] 第 6 步 wakeup_plan.json 生成

**全部 ✅ → 进 21 (IV sweep L1 空连)**
